# Pseudo-Inverse and Least Squares

Chapter 8: Singular Value Decomposition (SVD) · *Mathematics for AI and Machine Learning* by Fuheng Wu

Solve overdetermined systems via A^+ and visualize residuals.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-8/pseudoinverse-ls/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import torch

torch.set_printoptions(precision=4)

# The demo's fixed 3x2 matrix and default right-hand side
A = torch.tensor([[1.0, 0.2],
                  [0.7, 1.3],
                  [1.4, -0.6]], dtype=torch.float64)
b = torch.tensor([1.4, -0.8, 2.1], dtype=torch.float64)

# ── Three equal ways to get the least-squares x ─────────────────────────────
x_pinv = torch.linalg.pinv(A) @ b                         # x = A^+ b
x_normal = torch.linalg.solve(A.T @ A, A.T @ b)           # (A^T A) x = A^T b
U, S, Vh = torch.linalg.svd(A, full_matrices=False)
x_svd = Vh.T @ ((U.T @ b) / S)                            # A^+ = V Sigma^-1 U^T

print("x (pinv)   =", x_pinv.tolist())
print("x (normal) =", x_normal.tolist())
print("x (SVD)    =", x_svd.tolist())

# 3 equations, 2 unknowns: no exact solution, so a residual remains
r = A @ x_pinv - b
print("\nresidual r = Ax - b =", r.tolist(), " ||r|| =", r.norm().item())
print("A^T r (should be 0) =", (A.T @ r).tolist())

# ── A^+ satisfies the four Penrose conditions ───────────────────────────────
P = torch.linalg.pinv(A)
print("\nA A+ A = A  :", torch.allclose(A @ P @ A, A))
print("A+ A A+ = A+:", torch.allclose(P @ A @ P, P))
print("(A A+)^T = A A+:", torch.allclose((A @ P).T, A @ P))
print("A+ A = I (full column rank):", torch.allclose(P @ A, torch.eye(2, dtype=A.dtype)))

x (pinv)   = [1.1832548755884333, -1.1193678547410895]
x (normal) = [1.183254875588433, -1.1193678547410892]
x (SVD)    = [1.1832548755884331, -1.1193678547410895]

residual r = Ax - b = [-0.4406186953597845, 0.17310020174848695, 0.22817753866846013]  ||r|| = 0.5255221248422582
A^T r (should be 0) = [5.570497362345197e-16, 5.502659525077247e-17]

A A+ A = A  : True
A+ A A+ = A+: True
(A A+)^T = A A+: True
A+ A = I (full column rank): True
